## Book Histogram - Kernel Authoring

### Table of Contents

1. [Environment Setup & Data Download](#1.-Environment-Setup-&-Data-Download)
2. [First Attempt: Global Memory Histogram](#2.-First-Attempt:-Global-Memory-Histogram)
3. [Fixing Data Races with Atomics](#3.-Fixing-Data-Races-with-Atomics)
4. [Profiling the Naive Solution](#4.-Profiling-the-Naive-Solution)
5. [Optimization: Shared Memory & Cooperative Groups](#5.-Optimization:-Shared-Memory-&-Cooperative-Groups)
6. [Performance Comparison](#6.-Performance-Comparison)

### 1. Environment Setup & Data Download

Let's learn to use some advanced CUDA features like shared memory, atomics, and [cuda.cooperative](https://nvidia.github.io/cccl/unstable/python/coop.html) to write an efficient histogram kernel to determine the most frequent characters in a collection of books.

First, let's download our dataset and install the necessary tools.

In [ ]:
import os

# Install necessary packages if running in Google Colab.
if os.getenv("COLAB_RELEASE_TAG") and not os.path.exists("/accelerated-computing-hub-installed"):
  print("Downloading NCU package.")
  !curl -s -L -O https://developer.download.nvidia.com/compute/cuda/repos/debian12/x86_64/nsight-compute-2025.2.1_2025.2.1.3-1_amd64.deb
  print("Installing NCU package.")
  !dpkg -i nsight-compute-2025.2.1_2025.2.1.3-1_amd64.deb > /dev/null
  !update-alternatives --install /opt/bin/ncu ncu /opt/nvidia/nsight-compute/2025.2.1/ncu 20250201 > /dev/null
  print("Uninstalling PIP packages.")
  !pip uninstall "cuda-python" --yes > /dev/null
  print("Installing PIP packages.")
  !pip install "numba-cuda" "cuda-cccl[test-cu12]" "nvtx" "nsightful[notebook] @ git+https://github.com/brycelelbach/nsightful.git@1cdeaab47f823aac6e25b05251420f540ab0db42" > /dev/null 2>&1
  open("/accelerated-computing-hub-installed", "a").close()
  print("All packages installed.")

import numpy as np
import urllib.request
from jupyter_dark_detect import is_dark
import matplotlib.pyplot as plt
plt.style.use('dark_background' if is_dark() else 'default')
from numba import cuda
import cupy as cp
import cupyx as cpx


In [ ]:
urllib.request.urlretrieve(
  "https://drive.usercontent.google.com/download?id=1MW1lPgkTq3YG9ikuq6u3d9sfpt-wKQZ0&export=download",
  "books__15m.txt")

### 2. First Attempt: Global Memory Histogram

A histogram kernel counts the number of times a value occurs in a dataset. To implement this, we create an array that is large enough to store all possible values (in the case of counting 1-byte ASCII characters, 256 elements). Then for the value of each element in the dataset, we increment its location in the array.

Let's try a simple way to implement this:

In [ ]:
bins = 256
values = cp.fromfile("books__15m.txt", dtype=cp.uint8)
histogram = cp.zeros(bins, dtype=cp.int32)
threads_per_block = 512
items_per_thread = 8
items_per_block = threads_per_block * items_per_thread
blocks = len(values) // items_per_block
assert values.size % items_per_block == 0

@cuda.jit
def histogram_global(values, histogram):
  for i in range(items_per_thread):
    value = values[cuda.grid(1) * items_per_thread + i]
    old_count = histogram[value]
    new_count = old_count + 1
    histogram[value] = new_count


def launch_global():
  histogram[:] = 0
  histogram_global[blocks, threads_per_block](values, histogram)


Now let's make sure it runs and check the output.

In [ ]:
launch_global()
cp.cuda.runtime.deviceSynchronize()


In [ ]:
histogram_host = cp.asnumpy(histogram)

# Print most frequently occurring characters.
pairs = sorted(((i, c) for i, c in enumerate(histogram_host) if c), key=lambda x: x[1], reverse=True)[:20]
labels = [('SPACE' if i == 32 else chr(i)) if 32 <= i <= 126 else f'0x{i:02X}' for i, _ in pairs]
plt.barh(labels[::-1], [c for _, c in pairs][::-1])
plt.xlabel('count')
plt.tight_layout()
plt.title("Top 20 Bins")
plt.show()

print(f"Characters in dataset: {values.size / 1e6:.1f} MB")


### 3. Fixing Data Races with Atomics

It looks like something is wrong - our counts are very low, and the most common characters don't make a lot of sense. Many of our increments seem to get lost!

What's happening here is called a data race. Many different threads are trying to access the bins of the histogram at the same time.

Imagine that two threads are trying to update the same bin:

- Thread 0 reads the count of the bin, which is 0, and stores it in its local variable `old_count`.
- Thread 0 adds 1 to its `old_count`, producing a `new_count` of 1.
- Thread 1 reads the count of the bin, which is still 0, and stores it in its local variable `old_count`.
- Thread 1 adds 1 to its `old_count`, producing a `new_count` of 1.
- Thread 0 stores `new_count` to the bin, setting it to 1.
- Thread 1 stores `new_count` to the bin, setting it to 1, and losing the increment from thread 0!

To fix this, we need to use atomic operations. `cuda.atomic.add(array, index, value)` will perform `array[index] += value` as a single indivisible operation. This will ensure that no increments get lost.

**TODO: Fix the code above by modifying it to use `cuda.atomic.add()`.**

### 4. Profiling the Naive Solution

Select the **Python 3 (Nsight Compute)** kernel, then profile the naive kernel in place. The `%%ncu` magic preserves the arrays and compiled kernel from the earlier cells and displays the report below.

In [ ]:
%%ncu -o histogram_global.ncu-rep
histogram[:] = 0
histogram_global[blocks, threads_per_block](values, histogram)
cp.cuda.runtime.deviceSynchronize()

In [ ]:
assert cp.sum(histogram) < len(values), "The intentionally racy kernel should lose updates."

### 5. Optimization: Shared Memory & Cooperative Groups

Looking at the profile trace, it seems like our code is quite slow - look at the memory workload tab and see how low the throughput is!

One improvement we should make is to separate loading values from the histogram update and to perform striped loads (also known as coalesced access). We'll use [cuda.cooperative](https://nvidia.github.io/cccl/unstable/python/coop.html)'s block load instead of writing this by hand.

**TODO: Rewrite the code below to use `cuda.cooperative` to load from `values` into local memory.**
- **Create a `coop.block.load(dtype, threads_per_block, items_per_thread, algorithm)` object outside of the kernel.**
- **Make sure to link the algorithm object to the kernel by adding a `link` parameter to the decorator.**
- **Create storage for the items we'll load with `cuda.local.array()`.**

**TODO: Look at the profile trace and code and think about how we could improve performance further.**

**HINT:**
- **What sorts of operations are we performing? Are they expensive? Can we make the code more efficient by reducing the number of expensive operations we perform?**
- **You can allocate memory accessible by the entire block with `cuda.shared.array()`.**
- **You can synchronize all threads within a block with `cuda.syncthreads()`.**

In [ ]:
localized_items_per_thread = 8
localized_items_per_block = threads_per_block * localized_items_per_thread
localized_blocks = len(values) // localized_items_per_block
assert values.size % localized_items_per_block == 0

@cuda.jit
def histogram_localized(values, histogram):
  for i in range(localized_items_per_thread):
    value = values[cuda.grid(1) * localized_items_per_thread + i]
    old_count = histogram[value]
    new_count = old_count + 1
    histogram[value] = new_count


def launch_localized():
  histogram[:] = 0
  histogram_localized[localized_blocks, threads_per_block](values, histogram)


Now let's run the code, then profile the optimized kernel with the same Nsight Compute kernel.

In [ ]:
launch_localized()
cp.cuda.runtime.deviceSynchronize()
assert cp.sum(histogram) == len(values)


In [ ]:
histogram_host = cp.asnumpy(histogram)
pairs = sorted(((i, c) for i, c in enumerate(histogram_host) if c), key=lambda x: x[1], reverse=True)[:20]
labels = [('SPACE' if i == 32 else chr(i)) if 32 <= i <= 126 else f'0x{i:02X}' for i, _ in pairs]
plt.barh(labels[::-1], [c for _, c in pairs][::-1])
plt.xlabel('count')
plt.tight_layout()
plt.title("Top 20 Bins")
plt.show()


In [ ]:
%%ncu -o histogram_localized.ncu-rep
histogram[:] = 0
histogram_localized[localized_blocks, threads_per_block](values, histogram)
cp.cuda.runtime.deviceSynchronize()

In [ ]:
assert cp.sum(histogram) == len(values)

### 6. Performance Comparison

Let's compare the execution time of our naive global memory implementation against our optimized shared memory implementation.

In [ ]:
global_times = cpx.profiler.benchmark(launch_global, n_repeat=15, n_warmup=4).gpu_times[0]
localized_times = cpx.profiler.benchmark(launch_localized, n_repeat=15, n_warmup=4).gpu_times[0]
histogram_global_duration = global_times.mean() * 1000
histogram_localized_duration = localized_times.mean() * 1000
speedup = histogram_global_duration / histogram_localized_duration

print(f"histogram_global:    {histogram_global_duration:.3g} ms")
print(f"histogram_localized: {histogram_localized_duration:.3g} ms")
print(f"histogram_localized speedup over histogram_global: {speedup:.2f}")
